## Evaluation Framework

In order to set up an evaluation, we need to consider what we are aiming to get. Down the line, the ESCO nodes found by our vector search will be used to build a prompt so that the Large Language Model can ask users to validate a number of skills and occupations found through vector search. For that reason, we don't want to focus so much on the precision (that is, reducing the amount of false positives), but rather on the recall (that is, reducing the amount of false negatives). 


On this premise, we design an evaluation method (recall@k) in which we consider how many correct nodes are found within the top k classes. In addition, we also consider the precision@k which tells us how many of the k retrieved nodes are correct.  The precision@k evaluates how many of the top k retrieved nodes are true positives, penalizing the use of larger sample sizes. The recall@k, instead, focuses on calculating how many of the total true positives in a given sample are retrieved in the top k classes. This actually penalizes lower values of k, as less nodes will be retrieved. Both of these metrics could be further refined by considering the score to be inversely proportional to the rank in which the correct node is found. However, since we're not concerned with the rank, we will use a 0-1 score.

In the case of occupations, since our test set includes only one correct node per sentences, the recall@k reduces to understanding whether the correct node is within the first k elements. However, multiple skill nodes can correspond to a single sentence, so that the recall at k really captures what percentage of the correct skills on average is retrieved within the first k elements.


In [10]:
from typing import List

def precision_at_k(prediction: List[List[str]], true: List[List[str]], k:int):
    """Calculates the average precision at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total retrieved nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.

    Returns:
        float: average precision at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_precision = 0
    for pred_list, true_val in zip(prediction, true):
        total_precision+=len(set(pred_list[:k]).intersection(set(true_val)))/k
    return total_precision/len(true)

def recall_at_k(prediction: List[List[str]], true: List[List[str]], k:int):
    """Calculates the average recall at k considering
    for each prediction the number of correct retrieved nodes
    divided by the number of total correct nodes.

    Args:
        prediction (List[List[str]]): list of 
            predicted lists, each with the corresponding
            nodes.
        true (List[List[str]]): list of the multiple true nodes 
            for each sample in the dataset.

    Returns:
        float: average recall at k over all the test set.
    """
    assert len(prediction) == len(true)
    total_recall = 0
    for pred_list, true_val in zip(prediction, true):
        total_recall+=len(set(pred_list[:k]).intersection(set(true_val)))/len(set(true_val))
    return total_recall/len(true)

## Evaluation goals

The objective of the evaluation is to choose which model and hyperparameters have the highest recall at k. For a given embedding model, the hyperparameters are the following:

1. **How to embed a node of the graph**: which combination of the fields guarantees the best performance when embedded.
2. **Score function**: which function should be used to retrieve the most similar nodes (cosine, l2 distance or scalar product).
3. **Using Maximal Marginal Relevance**: whether we should use MMR to get more diverse results.

We will use ChromaDB as a local vector store and get the ESCO data from a local csv file. We will restrict our evaluation to the gecko003 model, but this can be repeated with any other model.

The evaluation will be conducted as follows:

1. Evaluation of the occupation nodes from the occupation dataset.
2. Evaluation of the related skill nodes from the occupation dataset.
3. Evaluation of the skill nodes from the skill dataset.

The corresponding datasets will be loaded from huggingface and a series of functions useful to all three evaluations will be defined before proceeding.

In [11]:
# 1. Loading the test dataset for occupations using the Huggingface library
from huggingface_hub import hf_hub_download
import pandas as pd
from tqdm import tqdm
import os 
from dotenv import load_dotenv
from vertexai.language_models import TextEmbeddingModel

tqdm.pandas()

load_dotenv("/Users/francescopreta/coding/compass/backend/.env")

HF_TOKEN = os.environ["HF_ACCESS_TOKEN"]
OCCUPATION_REPO_ID = "tabiya/hahu_test"
OCCUPATION_FILENAME = "redacted_hahu_test_with_id.csv"
SKILL_REPO_ID = "tabiya/esco_skills_test"
SKILL_FILENAME = "data/processed_skill_test_set_with_id.parquet"
OCCUPATION_CSV_PATH = "/Users/francescopreta/coding/compass/backend/esco_search/_scripts/occupations.csv"
SKILL_CSV_PATH = "/Users/francescopreta/coding/compass/backend/esco_search/_scripts/skills.csv"

df_occupation_test = pd.read_csv(
    hf_hub_download(repo_id=OCCUPATION_REPO_ID, filename=OCCUPATION_FILENAME, repo_type="dataset", token=HF_TOKEN)
)
df_skill_test = pd.read_parquet(
    hf_hub_download(repo_id=SKILL_REPO_ID, filename=SKILL_FILENAME, repo_type="dataset", token=HF_TOKEN)
)
df_occupation_database = pd.read_csv(OCCUPATION_CSV_PATH)
df_skill_database = pd.read_csv(SKILL_CSV_PATH)

model = TextEmbeddingModel.from_pretrained("textembedding-gecko@003")

In [3]:
# Normalize skills and occupation test target fields with the database targets
df_skill_database = df_skill_database.rename(columns={"UUIDHISTORY":"CODE"})
df_skill_test["CODE"] = df_skill_test["UUID"].apply(lambda x: [x])
df_occupation_test["CODE"] = df_occupation_test["esco_code"].apply(lambda x: [x])
df_occupation_test["skills_essential"] = df_occupation_test["skills_essential"].apply(eval)
df_occupation_test["skills_optional"] = df_occupation_test["skills_optional"].apply(eval)


In what follows, we will pre-compute the strings and the corresponding embeddings using the Gecko model. We will use manual batching to speed up the process, as the vertex API doesn't support batching and fails if the list length is larger than 250 elements or the sum of tokens is higher than 20.000.

In [4]:
def embed_strings_in_batch(list_of_queries: List[str], model: TextEmbeddingModel, batch_size: int = 100) -> List[List[float]]:
    """Uses a TextEmbeddingModel to embed a list of queries in batches.

    Args:
        list_of_queries (List[str]): list of queries to be embedded in batches.
        model (TextEmbeddingModel): embedding model.
        batch_size (int, optional): size of each batch which should be less than or equal to 250.
            Defaults to 100.

    Returns:
        List[List[float]]: List of embeddings corresponding to the queries.
    """
    assert batch_size<=250
    embedding_results = []
    num_samples = len(list_of_queries)
    for i in range(int(num_samples/batch_size)+1):
        batch = list_of_queries[i*batch_size:(i+1)*batch_size]
        embedding_results += model.get_embeddings(batch)
    assert len(embedding_results) == len(list_of_queries)
    return [embedding_result.values for embedding_result in embedding_results]

We define the functions generating the string that can be embedded from each document and save their computation in a dictionary. Then we save the corresponding embeddings in another dictionary, as well as the embeddings of the test set, that will be used for query retrieval.

In [5]:
# Functions defining strings to embed
def description(df):
    return df["DESCRIPTION"]

def preferred_label(df):
    return df["PREFERREDLABEL"]

def all_occupations(df):
    return f"""Occupation Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Occupation Description: {df['DESCRIPTION']}"""

def all_skills(df):
    return f"""Skill Names: {df['PREFERREDLABEL']}
{df['ALTLABELS']}

Skill Description: {df['DESCRIPTION']}"""

def label_and_description(df):
    return f"{df['PREFERREDLABEL']}\n{df['DESCRIPTION']}"

In [6]:
# Embedding precomputation
from typing import Any, Dict, Tuple

def precompute_embeddings(df_database: pd.DataFrame, function_to_method: Dict[str,Any]) -> pd.DataFrame:
    """For a given database and map that sends each function name to
    its respective function for selecting a substring from the node,
    returns an updated dataframe with the corresponding methods as
    well as embeddings for all the methods

    Args:
        df_database (pd.DataFrame): database of interest
        function_to_method (Dict[str,Any]): map from function
            name to function selecting string for any given node.

    Returns:
        The updated dataframe with the method strings and the corresponding
        embeddings.
    """
    for method in function_to_method:
        df_database[method] = df_database.progress_apply(function_to_method[method], axis=1)
        embeddings = embed_strings_in_batch(list(df_database[method]), model)
        df_database[f'embeddings_{method}'] = embeddings
    return df_database

function_to_occupation_method = {"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_OCCUPATIONS":all_occupations, "LABEL_AND_DESCRIPTION": label_and_description}
function_to_skill_method = {"DESCRIPTION": description, "PREFERREDLABEL":preferred_label, "ALL_SKILLS":all_skills, "LABEL_AND_DESCRIPTION": label_and_description}

# Compute database embeddings
df_occupation_database = precompute_embeddings(df_occupation_database, function_to_occupation_method)
df_skill_database = precompute_embeddings(df_skill_database, function_to_skill_method)

# Compute test set embeddings
test_occupation_embeddings = embed_strings_in_batch(list(df_occupation_test["synthetic_query"]), model)
df_occupation_test["embeddings"] = test_occupation_embeddings

test_skill_embeddings = embed_strings_in_batch(list(df_skill_test["synthetic_query"]), model)
df_skill_test["embeddings"] = test_skill_embeddings

  0%|          | 0/3007 [00:00<?, ?it/s]

100%|██████████| 13896/13896 [00:00<00:00, 324189.30it/s]


In [7]:
# Functions "maximal_marginal_relevance" and "cosine_similarity"
# are duplicated respectively from modules:
#    - "libs/community/langchain_community/vectorstores/utils.py"
#    - "libs/community/langchain_community/utils/math.py"
from typing import List, Union

import numpy as np

Matrix = Union[List[List[float]], List[np.ndarray], np.ndarray]


def cosine_similarity(X: Matrix, Y: Matrix) -> np.ndarray:
    """Row-wise cosine similarity between two equal-width matrices."""
    if len(X) == 0 or len(Y) == 0:
        return np.array([])

    X = np.array(X)
    Y = np.array(Y)
    if X.shape[1] != Y.shape[1]:
        raise ValueError(
            f"Number of columns in X and Y must be the same. X has shape {X.shape} "
            f"and Y has shape {Y.shape}."
        )
    X_norm = np.linalg.norm(X, axis=1)
    Y_norm = np.linalg.norm(Y, axis=1)
    # Ignore divide by zero errors run time warnings as those are handled below.
    with np.errstate(divide="ignore", invalid="ignore"):
        similarity = np.dot(X, Y.T) / np.outer(X_norm, Y_norm)
    similarity[np.isnan(similarity) | np.isinf(similarity)] = 0.0
    return similarity



def maximal_marginal_relevance(
    query_embedding: np.ndarray,
    embedding_list: list,
    lambda_mult: float = 0.5,
    k: int = 10,
) -> List[int]:
    """Calculate maximal marginal relevance."""
    if min(k, len(embedding_list)) <= 0:
        return []
    if query_embedding.ndim == 1:
        query_embedding = np.expand_dims(query_embedding, axis=0)
    similarity_to_query = cosine_similarity(query_embedding, embedding_list)[0]
    most_similar = int(np.argmax(similarity_to_query))
    idxs = [most_similar]
    selected = np.array([embedding_list[most_similar]])
    while len(idxs) < min(k, len(embedding_list)):
        best_score = -np.inf
        idx_to_add = -1
        similarity_to_selected = cosine_similarity(embedding_list, selected)
        for i, query_score in enumerate(similarity_to_query):
            if i in idxs:
                continue
            redundant_score = max(similarity_to_selected[i])
            equation_score = (
                lambda_mult * query_score - (1 - lambda_mult) * redundant_score
            )
            if equation_score > best_score:
                best_score = equation_score
                idx_to_add = i
        idxs.append(idx_to_add)
        selected = np.append(selected, [embedding_list[idx_to_add]], axis=0)
    return idxs


We create multiple chromadb collections to store our data in memory with different embeddings depending on the method used and on the function used for querying. On these, we save the Occupation ESCO database with all their metadatas.

In [8]:
import chromadb

SCORE_FUNCTIONS = ["cosine", "l2", "ip"]
client = chromadb.Client()

def create_collections(db_name: str, methods: List[str], df_database: pd.DataFrame):
    """Creates multiple collections for each choice of db_name
    and corresponding documents and embeddings.

    Args:
        db_name (str): name of the database. 
            Either 'skills' or 'occupations'.
        methods (List[str]): list of methods for the embeddings.
        df_database (pd.DataFrame): database containing the metadata.
    """
    for method in methods:
        for score in SCORE_FUNCTIONS:
            collection_name = f'{db_name}_{method}_{score}'
            collection_metadata = {"hnsw:space":score}
            collection = client.create_collection(name=collection_name, metadata=collection_metadata)
            collection.add(
                documents = list(df_database[method]),
                metadatas = [{"CODE": row["CODE"]} for _, row in df_database.iterrows()],
                embeddings = list(df_database[f"embeddings_{method}"]),
                ids = [f"id_{i}" for i in range(len(df_database))]
            )

In [9]:
create_collections("occupations", list(function_to_occupation_method.keys()), df_occupation_database)
create_collections("skills", list(function_to_skill_method.keys()), df_skill_database)

Finally, we write a function to run the evaluation as follows:

1. We choose a score function and a method and load the corresponding collection.
2. For each element in the test set, we find the top 100 documents in the collection ordered by scoring rank.
3. We filter those documents by maximal marginal relevance to find the top 10 documents with this function.
4. We evaluate the precision and recall on the top k for k=1,3,5,10 for the original retrieved documents.
5. We evaluate the accuracy on the top k for k=1,3,5,10 for the documents filtered by maximal marginal relevance.
6. We save the results in a dataframe to be analyzed for later use.

In [12]:
def run_eval(db_type: str, method_list: List[str], df_test: pd.DataFrame, test_target: str = "CODE") -> pd.DataFrame:
    """Returns the results of an evaluation on a list of collections

    Args:
        db_type (str): name of the database (occupation or skill).
        collection_name_list (List[str]): list of collections of interest
            to test.
        df_test (pd.DataFrame): test dataframe, containing an embedding column
            and a test_target column.
        test_target (str): name of the target field. Defaults to 'CODE'.

    Returns:
        pd.DataFrame: dataframe with the result of the evaluation depending on the
            different hyperparameters.
    """
    eval_data = []
    for method in method_list:
        for score in SCORE_FUNCTIONS:
            collection_name = f"{db_type}_{method}_{score}"
            # Fetch collection
            collection = client.get_collection(name=collection_name)
            # Initialize lists to save results
            vector_search_results = []
            mmr_vector_search_results = []
            for test_embedding in list(df_test["embeddings"]):
                # Find the top 100 documents and save them in vector_search_results
                documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
                vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
                # Find the top 10 documents according to MMR and save them in mmr_vector_search_results
                result_embeddings = [elem for elem in documents_from_search["embeddings"][0]]
                mmr_ids = maximal_marginal_relevance(np.array(test_embedding), result_embeddings)
                mmr_vector_search_results.append([elem["CODE"] for index, elem in enumerate(documents_from_search["metadatas"][0]) if index in mmr_ids])
            # Evaluate accuracy at k for k=1, 3, 5, 10
            for k in [1, 3, 5, 10]:
                rec_at_k = recall_at_k(vector_search_results, list(df_test[test_target]), k)
                prec_at_k = precision_at_k(vector_search_results, list(df_test[test_target]), k)
                eval_data.append({"method":method, "score function":score, "MMR": False, "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4)})
                rec_at_k = recall_at_k(mmr_vector_search_results, list(df_test[test_target]), k)
                prec_at_k = precision_at_k(mmr_vector_search_results, list(df_test[test_target]), k)
                eval_data.append({"method":method, "score function":score, "MMR": True, "k":k, "recall": round(rec_at_k, 4), "precision": round(prec_at_k,4)})
    # Save the results in a dataframe
    eval_df = pd.DataFrame(eval_data)
    return eval_df

We can now run all the evaluations, where the occupation evaluation will use the occupation databases, while the skills

In [13]:
# Save the evaluation results locally
OUTPUT_EVAL_PATH = "/Users/francescopreta/coding/compass/backend/esco_search/_scripts/"

# Evaluation of occupation
df_occupation_eval = run_eval("occupations", list(function_to_occupation_method.keys()), df_occupation_test)
df_occupation_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "occupation_eval.csv"))

# Evaluation of skill linking to occupation
df_skill_occupation_eval = run_eval("skills", list(function_to_skill_method.keys()), df_occupation_test, "skills_essential")
df_skill_occupation_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "occupation_skills_eval.csv"))

# Evaluation of skills
df_skill_eval = run_eval("skills", list(function_to_skill_method.keys()), df_skill_test)
df_skill_eval.to_csv(os.path.join(OUTPUT_EVAL_PATH, "skill_eval.csv"))


Let's now discuss the results of our experiments

### Occupations

The following table illustrates the result of our experiments:

| method               | score function | MMR  | k   | recall | precision |
|----------------------|----------------|------|-----|--------|-----------|
| PREFERREDLABEL       | cosine         | False| 10  | 0.7454 | 0.0745    |
| PREFERREDLABEL       | l2             | False| 10  | 0.7454 | 0.0745    |
| PREFERREDLABEL       | ip             | False| 10  | 0.7454 | 0.0745    |
| ALL_OCCUPATIONS      | cosine         | False| 10  | 0.738  | 0.0738    |
| ALL_OCCUPATIONS      | l2             | False| 10  | 0.738  | 0.0738    |
| ALL_OCCUPATIONS      | ip             | False| 10  | 0.738  | 0.0738    |
| LABEL_AND_DESCRIPTION| cosine         | False| 10  | 0.7196 | 0.072     |
| LABEL_AND_DESCRIPTION| l2             | False| 10  | 0.7196 | 0.072     |
| LABEL_AND_DESCRIPTION| ip             | False| 10  | 0.7196 | 0.072     |
| DESCRIPTION          | cosine         | False| 10  | 0.714  | 0.0714    |
| DESCRIPTION          | l2             | False| 10  | 0.714  | 0.0714    |
| DESCRIPTION          | ip             | False| 10  | 0.714  | 0.0714    |
| PREFERREDLABEL       | cosine         | True | 10  | 0.4502 | 0.045     |
| PREFERREDLABEL       | l2             | True | 10  | 0.4502 | 0.045     |
| PREFERREDLABEL       | ip             | True | 10  | 0.4502 | 0.045     |
| ALL_OCCUPATIONS      | cosine         | True | 10  | 0.4299 | 0.043     |
| ALL_OCCUPATIONS      | l2             | True | 10  | 0.4299 | 0.043     |
| ALL_OCCUPATIONS      | ip             | True | 10  | 0.4299 | 0.043     |
| LABEL_AND_DESCRIPTION| cosine         | True | 10  | 0.4244 | 0.0424    |
| LABEL_AND_DESCRIPTION| l2             | True | 10  | 0.4244 | 0.0424    |
| LABEL_AND_DESCRIPTION| ip             | True | 10  | 0.4244 | 0.0424    |
| DESCRIPTION          | cosine         | True | 10  | 0.4059 | 0.0406    |
| DESCRIPTION          | l2             | True | 10  | 0.4059 | 0.0406    |
| DESCRIPTION          | ip             | True | 10  | 0.4059 | 0.0406    |
| PREFERREDLABEL       | cosine         | False| 5   | 0.6624 | 0.1325    |
| PREFERREDLABEL       | l2             | False| 5   | 0.6624 | 0.1325    |
| PREFERREDLABEL       | ip             | False| 5   | 0.6624 | 0.1325    |
| ALL_OCCUPATIONS      | cosine         | False| 5   | 0.6605 | 0.1321    |
| ALL_OCCUPATIONS      | l2             | False| 5   | 0.6605 | 0.1321    |
| ALL_OCCUPATIONS      | ip             | False| 5   | 0.6605 | 0.1321    |
| DESCRIPTION          | cosine         | False| 5   | 0.6199 | 0.124     |
| DESCRIPTION          | l2             | False| 5   | 0.6199 | 0.124     |
| DESCRIPTION          | ip             | False| 5   | 0.6199 | 0.124     |
| LABEL_AND_DESCRIPTION| cosine         | False| 5   | 0.6144 | 0.1229    |
| LABEL_AND_DESCRIPTION| l2             | False| 5   | 0.6144 | 0.1229    |
| LABEL_AND_DESCRIPTION| ip             | False| 5   | 0.6144 | 0.1229    |
| PREFERREDLABEL       | cosine         | True | 5   | 0.4446 | 0.0889    |
| PREFERREDLABEL       | l2             | True | 5   | 0.4446 | 0.0889    |
| PREFERREDLABEL       | ip             | True | 5   | 0.4446 | 0.0889    |
| ALL_OCCUPATIONS      | cosine         | True | 5   | 0.4244 | 0.0849    |
| ALL_OCCUPATIONS      | l2             | True | 5   | 0.4244 | 0.0849    |
| ALL_OCCUPATIONS      | ip             | True | 5   | 0.4244 | 0.0849    |
| LABEL_AND_DESCRIPTION| cosine         | True | 5   | 0.4244 | 0.0849    |
| LABEL_AND_DESCRIPTION| l2             | True | 5   | 0.4244 | 0.0849    |
| LABEL_AND_DESCRIPTION| ip             | True | 5   | 0.4244 | 0.0849    |
| DESCRIPTION          | cosine         | True | 5   | 0.4022 | 0.0804    |
| DESCRIPTION          | l2             | True | 5   | 0.4022 | 0.0804    |
| DESCRIPTION          | ip             | True | 5   | 0.4022 | 0.0804    |
| PREFERREDLABEL       | cosine         | False| 3   | 0.5812 | 0.1937    |
| PREFERREDLABEL       | l2             | False| 3   | 0.5812 | 0.1937    |
| PREFERREDLABEL       | ip             | False| 3   | 0.5812 | 0.1937    |
| ALL_OCCUPATIONS      | cosine         | False| 3   | 0.5812 | 0.1937    |
| ALL_OCCUPATIONS      | l2             | False| 3   | 0.5812 | 0.1937    |
| ALL_OCCUPATIONS      | ip             | False| 3   | 0.5812 | 0.1937    |
| LABEL_AND_DESCRIPTION| cosine         | False| 3   | 0.548  | 0.1827    |
| LABEL_AND_DESCRIPTION| l2             | False| 3   | 0.548  | 0.1827    |
| LABEL_AND_DESCRIPTION| ip             | False| 3   | 0.548  | 0.1827    |
| DESCRIPTION          | cosine         | False| 3   | 0.5443 | 0.1814    |
| DESCRIPTION          | l2             | False| 3   | 0.5443 | 0.1814    |
| DESCRIPTION          | ip             | False| 3   | 0.5443 | 0.1814    |
| PREFERREDLABEL       | cosine         | True | 3   | 0.4299 | 0.1433    |
| PREFERREDLABEL       | l2             | True | 3   | 0.4299 | 0.1433    |
| PREFERREDLABEL       | ip             | True | 3   | 0.4299 | 0.1433    |
| ALL_OCCUPATIONS      | cosine         | True | 3   | 0.4151 | 0.1384    |
| ALL_OCCUPATIONS      | l2             | True | 3   | 0.4151 | 0.1384    |
| ALL_OCCUPATIONS      | ip             | True | 3   | 0.4151 | 0.1384    |
| LABEL_AND_DESCRIPTION| cosine         | True | 3   | 0.4151 | 0.1384    |
| LABEL_AND_DESCRIPTION| l2             | True | 3   | 0.4151 | 0.1384    |
| LABEL_AND_DESCRIPTION| ip             | True | 3   | 0.4151 | 0.1384    |
| DESCRIPTION          | cosine         | True | 3   | 0.3875 | 0.1292    |
| DESCRIPTION          | l2             | True | 3   | 0.3875 | 0.1292    |
| DESCRIPTION          | ip             | True | 3   | 0.3875 | 0.1292    |
| PREFERREDLABEL       | cosine         | False| 1   | 0.3801 | 0.3801    |
| PREFERREDLABEL       | cosine         | True | 1   | 0.3801 | 0.3801    |
| PREFERREDLABEL       | l2             | False| 1   | 0.3801 | 0.3801    |
| PREFERREDLABEL       | l2             | True | 1   | 0.3801 | 0.3801    |
| PREFERREDLABEL       | ip             | False| 1   | 0.3801 | 0.3801    |
| PREFERREDLABEL       | ip             | True | 1   | 0.3801 | 0.3801    |
| LABEL_AND_DESCRIPTION| cosine         | False| 1   | 0.3727 | 0.3727    |
| LABEL_AND_DESCRIPTION| cosine         | True | 1   | 0.3727 | 0.3727    |
| LABEL_AND_DESCRIPTION| l2             | False| 1   | 0.3727 | 0.3727    |
| LABEL_AND_DESCRIPTION| l2             | True | 1   | 0.3727 | 0.3727    |
| LABEL_AND_DESCRIPTION| ip             | False| 1   | 0.3727 | 0.3727    |
| LABEL_AND_DESCRIPTION| ip             | True | 1   | 0.3727 | 0.3727    |
| ALL_OCCUPATIONS      | cosine         | False| 1   | 0.3469 | 0.3469    |
| ALL_OCCUPATIONS      | cosine         | True | 1   | 0.3469 | 0.3469    |
| ALL_OCCUPATIONS      | l2             | False| 1   | 0.3469 | 0.3469    |
| ALL_OCCUPATIONS      | l2             | True | 1   | 0.3469 | 0.3469    |
| ALL_OCCUPATIONS      | ip             | False| 1   | 0.3469 | 0.3469    |
| ALL_OCCUPATIONS      | ip             | True | 1   | 0.3469 | 0.3469    |
| DESCRIPTION          | cosine         | False| 1   | 0.3413 | 0.3413    |
| DESCRIPTION          | cosine         | True | 1   | 0.3413 | 0.3413    |
| DESCRIPTION          | l2             | False| 1   | 0.3413 | 0.3413    |
| DESCRIPTION          | l2             | True | 1   | 0.3413 | 0.3413    |
| DESCRIPTION          | ip             | False| 1   | 0.3413 | 0.3413    |
| DESCRIPTION          | ip             | True | 1   | 0.3413 | 0.3413    |

The result of the evaluation are as follows:

1. The results obtained without MMR are definitely better than the results obtained without MMR. This happens because the correct code is most of the times within the first k elements and still very similar to the first one. MMR excludes many good high ranking results that could be retrieved otherwise because they are too similar to the first result.
2. Different retrieval functions return the same results. This probably happens because the Gecko embeddings are normalized so that l2, cosine similarity and dot product all determine the same ranking. We will choose the default version.
3. The best embedding methods is PREFERREDLABEL, slightly higher than ALL_OCCUPATIONS in our k of interest (3 or 5). We are choosing PREFERREDLABEL given that it has a small margin over ALL_OCCUPATIONS and it's easier to implement as its value is already present in the dataset. We think that the information contained in such label helps the model identify the correct occupation even when they are not using the same words, as the correspondence between ESCO nodes and PREFERREDLABEL is one-to-one. On the other hand, a secondary labels can appear in multiple nodes, thus confusing the model.

### Skills

| method               | score function | MMR  | k   | recall | precision |
|----------------------|----------------|------|-----|--------|-----------|
| PREFERREDLABEL       | cosine         | False| 10  | 0.4779 | 0.0478    |
| PREFERREDLABEL       | l2             | False| 10  | 0.4779 | 0.0478    |
| PREFERREDLABEL       | ip             | False| 10  | 0.4779 | 0.0478    |
| LABEL_AND_DESCRIPTION| cosine         | False| 10  | 0.4377 | 0.0438    |
| LABEL_AND_DESCRIPTION| l2             | False| 10  | 0.4377 | 0.0438    |
| LABEL_AND_DESCRIPTION| ip             | False| 10  | 0.4377 | 0.0438    |
| ALL_SKILLS           | cosine         | False| 10  | 0.4252 | 0.0425    |
| ALL_SKILLS           | l2             | False| 10  | 0.4252 | 0.0425    |
| ALL_SKILLS           | ip             | False| 10  | 0.4252 | 0.0425    |
| DESCRIPTION          | cosine         | False| 10  | 0.3348 | 0.0335    |
| DESCRIPTION          | l2             | False| 10  | 0.3348 | 0.0335    |
| DESCRIPTION          | ip             | False| 10  | 0.3348 | 0.0335    |
| PREFERREDLABEL       | cosine         | True | 10  | 0.3065 | 0.0307    |
| PREFERREDLABEL       | l2             | True | 10  | 0.306  | 0.0306    |
| PREFERREDLABEL       | ip             | True | 10  | 0.306  | 0.0306    |
| LABEL_AND_DESCRIPTION| cosine         | True | 10  | 0.2628 | 0.0263    |
| LABEL_AND_DESCRIPTION| l2             | True | 10  | 0.2623 | 0.0262    |
| LABEL_AND_DESCRIPTION| ip             | True | 10  | 0.2623 | 0.0262    |
| ALL_SKILLS           | l2             | True | 10  | 0.2529 | 0.0253    |
| ALL_SKILLS           | cosine         | True | 10  | 0.2519 | 0.0252    |
| ALL_SKILLS           | ip             | True | 10  | 0.2514 | 0.0251    |
| DESCRIPTION          | ip             | True | 10  | 0.1873 | 0.0187    |
| DESCRIPTION          | cosine         | True | 10  | 0.1868 | 0.0187    |
| DESCRIPTION          | l2             | True | 10  | 0.1868 | 0.0187    |
| PREFERREDLABEL       | cosine         | False| 5   | 0.3949 | 0.079     |
| PREFERREDLABEL       | l2             | False| 5   | 0.3949 | 0.079     |
| PREFERREDLABEL       | ip             | False| 5   | 0.3949 | 0.079     |
| ALL_SKILLS           | cosine         | False| 5   | 0.3502 | 0.07      |
| ALL_SKILLS           | l2             | False| 5   | 0.3502 | 0.07      |
| ALL_SKILLS           | ip             | False| 5   | 0.3502 | 0.07      |
| LABEL_AND_DESCRIPTION| cosine         | False| 5   | 0.3467 | 0.0693    |
| LABEL_AND_DESCRIPTION| l2             | False| 5   | 0.3467 | 0.0693    |
| LABEL_AND_DESCRIPTION| ip             | False| 5   | 0.3467 | 0.0693    |
| PREFERREDLABEL       | cosine         | True | 5   | 0.2906 | 0.0581    |
| PREFERREDLABEL       | l2             | True | 5   | 0.2906 | 0.0581    |
| PREFERREDLABEL       | ip             | True | 5   | 0.2906 | 0.0581    |
| DESCRIPTION          | cosine         | False| 5   | 0.2543 | 0.0509    |
| DESCRIPTION          | l2             | False| 5   | 0.2543 | 0.0509    |
| DESCRIPTION          | ip             | False| 5   | 0.2543 | 0.0509    |
| LABEL_AND_DESCRIPTION| cosine         | True | 5   | 0.2489 | 0.0498    |
| LABEL_AND_DESCRIPTION| l2             | True | 5   | 0.2484 | 0.0497    |
| LABEL_AND_DESCRIPTION| ip             | True | 5   | 0.2484 | 0.0497    |
| ALL_SKILLS           | l2             | True | 5   | 0.2449 | 0.049     |
| ALL_SKILLS           | cosine         | True | 5   | 0.2434 | 0.0487    |
| ALL_SKILLS           | ip             | True | 5   | 0.2434 | 0.0487    |
| DESCRIPTION          | ip             | True | 5   | 0.1783 | 0.0357    |
| DESCRIPTION          | cosine         | True | 5   | 0.1778 | 0.0356    |
| DESCRIPTION          | l2             | True | 5   | 0.1778 | 0.0356    |
| PREFERREDLABEL       | cosine         | False| 3   | 0.3313 | 0.1104    |
| PREFERREDLABEL       | l2             | False| 3   | 0.3313 | 0.1104    |
| PREFERREDLABEL       | ip             | False| 3   | 0.3313 | 0.1104    |
| ALL_SKILLS           | cosine         | False| 3   | 0.2951 | 0.0984    |
| ALL_SKILLS           | l2             | False| 3   | 0.2951 | 0.0984    |
| ALL_SKILLS           | ip             | False| 3   | 0.2951 | 0.0984    |
| LABEL_AND_DESCRIPTION| cosine         | False| 3   | 0.2891 | 0.0964    |
| LABEL_AND_DESCRIPTION| l2             | False| 3   | 0.2891 | 0.0964    |
| LABEL_AND_DESCRIPTION| ip             | False| 3   | 0.2891 | 0.0964    |
| PREFERREDLABEL       | cosine         | True | 3   | 0.2777 | 0.0926    |
| PREFERREDLABEL       | l2             | True | 3   | 0.2772 | 0.0924    |
| PREFERREDLABEL       | ip             | True | 3   | 0.2772 | 0.0924    |
| ALL_SKILLS           | l2             | True | 3   | 0.2305 | 0.0768    |
| LABEL_AND_DESCRIPTION| cosine         | True | 3   | 0.2305 | 0.0768    |
| LABEL_AND_DESCRIPTION| l2             | True | 3   | 0.2305 | 0.0768    |
| LABEL_AND_DESCRIPTION| ip             | True | 3   | 0.2305 | 0.0768    |
| ALL_SKILLS           | cosine         | True | 3   | 0.23   | 0.0767    |
| ALL_SKILLS           | ip             | True | 3   | 0.2295 | 0.0765    |
| DESCRIPTION          | cosine         | False| 3   | 0.2067 | 0.0689    |
| DESCRIPTION          | l2             | False| 3   | 0.2067 | 0.0689    |
| DESCRIPTION          | ip             | False| 3   | 0.2067 | 0.0689    |
| DESCRIPTION          | ip             | True | 3   | 0.1659 | 0.0553    |
| DESCRIPTION          | cosine         | True | 3   | 0.1654 | 0.0551    |
| DESCRIPTION          | l2             | True | 3   | 0.1654 | 0.0551    |
| PREFERREDLABEL       | cosine         | False| 1   | 0.2012 | 0.2012    |
| PREFERREDLABEL       | cosine         | True | 1   | 0.2012 | 0.2012    |
| PREFERREDLABEL       | l2             | False| 1   | 0.2012 | 0.2012    |
| PREFERREDLABEL       | l2             | True | 1   | 0.2012 | 0.2012    |
| PREFERREDLABEL       | ip             | False| 1   | 0.2012 | 0.2012    |
| PREFERREDLABEL       | ip             | True | 1   | 0.2012 | 0.2012    |
| LABEL_AND_DESCRIPTION| cosine         | False| 1   | 0.1664 | 0.1664    |
| LABEL_AND_DESCRIPTION| cosine         | True | 1   | 0.1664 | 0.1664    |
| LABEL_AND_DESCRIPTION| l2             | False| 1   | 0.1664 | 0.1664    |
| LABEL_AND_DESCRIPTION| l2             | True | 1   | 0.1664 | 0.1664    |
| LABEL_AND_DESCRIPTION| ip             | False| 1   | 0.1664 | 0.1664    |
| LABEL_AND_DESCRIPTION| ip             | True | 1   | 0.1664 | 0.1664    |
| ALL_SKILLS           | cosine         | False| 1   | 0.159  | 0.159     |
| ALL_SKILLS           | cosine         | True | 1   | 0.159  | 0.159     |
| ALL_SKILLS           | l2             | False| 1   | 0.159  | 0.159     |
| ALL_SKILLS           | l2             | True | 1   | 0.159  | 0.159     |
| ALL_SKILLS           | ip             | False| 1   | 0.159  | 0.159     |
| ALL_SKILLS           | ip             | True | 1   | 0.159  | 0.159     |
| DESCRIPTION          | cosine         | False| 1   | 0.1167 | 0.1167    |
| DESCRIPTION          | cosine         | True | 1   | 0.1167 | 0.1167    |
| DESCRIPTION          | l2             | False| 1   | 0.1167 | 0.1167    |
| DESCRIPTION          | l2             | True | 1   | 0.1167 | 0.1167    |
| DESCRIPTION          | ip             | False| 1   | 0.1167 | 0.1167    |
| DESCRIPTION          | ip             | True | 1   | 0.1167 | 0.1167    |

The result of the evaluation are similar to those for the occupations, that is:

1. The results obtained without MMR are definitely better than the results obtained without MMR. This happens because the correct code is most of the times within the first k elements and still very similar to the first one. MMR excludes many good high ranking results that could be retrieved otherwise because they are too similar to the first result.
2. Different retrieval functions return the same results. This probably happens because the Gecko embeddings are normalized so that l2, cosine similarity and dot product all determine the same ranking. We will choose the default version.
3. The best embedding methods is PREFERREDLABEL, higher than ALL_SKILLS in our k of interest (3 or 5). PREFERREDLABEL has a high margin over the second best and it's easier to implement as its value is already present in the dataset. We think that the information contained in such label helps the model identify the correct occupation even when they are not using the same words, as the correspondence between ESCO nodes and PREFERREDLABEL is one-to-one. On the other hand, a secondary labels can appear in multiple nodes, thus confusing the model.

### Occupation-related Skills

| method               | score function | MMR  | k   | recall | precision |
|----------------------|----------------|------|-----|--------|-----------|
| LABEL_AND_DESCRIPTION| cosine         | False| 10  | 0.0614 | 0.1491    |
| LABEL_AND_DESCRIPTION| l2             | False| 10  | 0.0614 | 0.1491    |
| LABEL_AND_DESCRIPTION| ip             | False| 10  | 0.0614 | 0.1491    |
| ALL_SKILLS           | cosine         | False| 10  | 0.0586 | 0.1426    |
| ALL_SKILLS           | l2             | False| 10  | 0.0586 | 0.1426    |
| ALL_SKILLS           | ip             | False| 10  | 0.0586 | 0.1426    |
| DESCRIPTION          | cosine         | False| 10  | 0.0573 | 0.1439    |
| DESCRIPTION          | l2             | False| 10  | 0.0573 | 0.1439    |
| DESCRIPTION          | ip             | False| 10  | 0.0573 | 0.1439    |
| PREFERREDLABEL       | cosine         | False| 10  | 0.0571 | 0.1413    |
| PREFERREDLABEL       | l2             | False| 10  | 0.0571 | 0.1413    |
| PREFERREDLABEL       | ip             | False| 10  | 0.0571 | 0.1413    |
| DESCRIPTION          | l2             | True | 10  | 0.0272 | 0.0686    |
| DESCRIPTION          | cosine         | True | 10  | 0.0271 | 0.0683    |
| DESCRIPTION          | ip             | True | 10  | 0.0271 | 0.0685    |
| LABEL_AND_DESCRIPTION| cosine         | True | 10  | 0.0265 | 0.0657    |
| LABEL_AND_DESCRIPTION| l2             | True | 10  | 0.0265 | 0.0659    |
| LABEL_AND_DESCRIPTION| ip             | True | 10  | 0.0265 | 0.0657    |
| PREFERREDLABEL       | cosine         | True | 10  | 0.0261 | 0.0653    |
| PREFERREDLABEL       | l2             | True | 10  | 0.0261 | 0.0653    |
| PREFERREDLABEL       | ip             | True | 10  | 0.0261 | 0.0655    |
| ALL_SKILLS           | l2             | True | 10  | 0.0233 | 0.0605    |
| ALL_SKILLS           | cosine         | True | 10  | 0.0232 | 0.0603    |
| ALL_SKILLS           | ip             | True | 10  | 0.0232 | 0.0603    |
| LABEL_AND_DESCRIPTION| cosine         | False| 5   | 0.0367 | 0.1753    |
| LABEL_AND_DESCRIPTION| l2             | False| 5   | 0.0367 | 0.1753    |
| LABEL_AND_DESCRIPTION| ip             | False| 5   | 0.0367 | 0.1753    |
| DESCRIPTION          | cosine         | False| 5   | 0.0334 | 0.1627    |
| DESCRIPTION          | l2             | False| 5   | 0.0334 | 0.1627    |
| DESCRIPTION          | ip             | False| 5   | 0.0334 | 0.1627    |
| PREFERREDLABEL       | cosine         | False| 5   | 0.0333 | 0.1624    |
| PREFERREDLABEL       | l2             | False| 5   | 0.0333 | 0.1624    |
| PREFERREDLABEL       | ip             | False| 5   | 0.0333 | 0.1624    |
| ALL_SKILLS           | cosine         | False| 5   | 0.0326 | 0.1542    |
| ALL_SKILLS           | l2             | False| 5   | 0.0326 | 0.1542    |
| ALL_SKILLS           | ip             | False| 5   | 0.0326 | 0.1542    |
| DESCRIPTION          | l2             | True | 5   | 0.0215 | 0.1044    |
| DESCRIPTION          | cosine         | True | 5   | 0.0214 | 0.1041    |
| DESCRIPTION          | ip             | True | 5   | 0.0214 | 0.1041    |
| LABEL_AND_DESCRIPTION| cosine         | True | 5   | 0.0198 | 0.0959    |
| LABEL_AND_DESCRIPTION| l2             | True | 5   | 0.0198 | 0.0959    |
| LABEL_AND_DESCRIPTION| ip             | True | 5   | 0.0198 | 0.0959    |
| PREFERREDLABEL       | cosine         | True | 5   | 0.0192 | 0.0952    |
| PREFERREDLABEL       | l2             | True | 5   | 0.0192 | 0.0952    |
| PREFERREDLABEL       | ip             | True | 5   | 0.0192 | 0.0952    |
| ALL_SKILLS           | cosine         | True | 5   | 0.0179 | 0.0919    |
| ALL_SKILLS           | l2             | True | 5   | 0.0179 | 0.0919    |
| ALL_SKILLS           | ip             | True | 5   | 0.0178 | 0.0915    |
| LABEL_AND_DESCRIPTION| cosine         | False| 3   | 0.0241 | 0.1907    |
| LABEL_AND_DESCRIPTION| l2             | False| 3   | 0.0241 | 0.1907    |
| LABEL_AND_DESCRIPTION| ip             | False| 3   | 0.0241 | 0.1907    |
| DESCRIPTION          | cosine         | False| 3   | 0.0224 | 0.179     |
| DESCRIPTION          | l2             | False| 3   | 0.0224 | 0.179     |
| DESCRIPTION          | ip             | False| 3   | 0.0224 | 0.179     |
| PREFERREDLABEL       | cosine         | False| 3   | 0.0209 | 0.1691    |
| PREFERREDLABEL       | l2             | False| 3   | 0.0209 | 0.1691    |
| PREFERREDLABEL       | ip             | False| 3   | 0.0209 | 0.1691    |
| ALL_SKILLS           | cosine         | False| 3   | 0.0205 | 0.1605    |
| ALL_SKILLS           | l2             | False| 3   | 0.0205 | 0.1605    |
| ALL_SKILLS           | ip             | False| 3   | 0.0205 | 0.1605    |
| DESCRIPTION          | cosine         | True | 3   | 0.0167 | 0.1322    |
| DESCRIPTION          | l2             | True | 3   | 0.0167 | 0.1335    |
| DESCRIPTION          | ip             | True | 3   | 0.0167 | 0.1328    |
| LABEL_AND_DESCRIPTION| ip             | True | 3   | 0.0158 | 0.1292    |
| LABEL_AND_DESCRIPTION| cosine         | True | 3   | 0.0157 | 0.1285    |
| LABEL_AND_DESCRIPTION| l2             | True | 3   | 0.0157 | 0.1285    |
| PREFERREDLABEL       | cosine         | True | 3   | 0.0148 | 0.1199    |
| PREFERREDLABEL       | l2             | True | 3   | 0.0148 | 0.1199    |
| PREFERREDLABEL       | ip             | True | 3   | 0.0148 | 0.1199    |
| ALL_SKILLS           | l2             | True | 3   | 0.0142 | 0.1187    |
| ALL_SKILLS           | cosine         | True | 3   | 0.0141 | 0.1187    |
| ALL_SKILLS           | ip             | True | 3   | 0.0141 | 0.1181    |
| DESCRIPTION          | cosine         | False| 1   | 0.0095 | 0.2103    |
| DESCRIPTION          | cosine         | True | 1   | 0.0095 | 0.2103    |
| DESCRIPTION          | l2             | False| 1   | 0.0095 | 0.2103    |
| DESCRIPTION          | l2             | True | 1   | 0.0095 | 0.2103    |
| DESCRIPTION          | ip             | False| 1   | 0.0095 | 0.2103    |
| DESCRIPTION          | ip             | True | 1   | 0.0095 | 0.2103    |
| LABEL_AND_DESCRIPTION| cosine         | False| 1   | 0.0089 | 0.203     |
| LABEL_AND_DESCRIPTION| cosine         | True | 1   | 0.0089 | 0.203     |
| LABEL_AND_DESCRIPTION| l2             | False| 1   | 0.0089 | 0.203     |
| LABEL_AND_DESCRIPTION| l2             | True | 1   | 0.0089 | 0.203     |
| LABEL_AND_DESCRIPTION| ip             | False| 1   | 0.0089 | 0.203     |
| LABEL_AND_DESCRIPTION| ip             | True | 1   | 0.0089 | 0.203     |
| PREFERREDLABEL       | cosine         | False| 1   | 0.0085 | 0.1845    |
| PREFERREDLABEL       | cosine         | True | 1   | 0.0085 | 0.1845    |
| PREFERREDLABEL       | l2             | False| 1   | 0.0085 | 0.1845    |
| PREFERREDLABEL       | l2             | True | 1   | 0.0085 | 0.1845    |
| PREFERREDLABEL       | ip             | False| 1   | 0.0085 | 0.1845    |
| PREFERREDLABEL       | ip             | True | 1   | 0.0085 | 0.1845    |
| ALL_SKILLS           | cosine         | False| 1   | 0.0074 | 0.1771    |
| ALL_SKILLS           | cosine         | True | 1   | 0.0074 | 0.1771    |
| ALL_SKILLS           | l2             | False| 1   | 0.0074 | 0.1771    |
| ALL_SKILLS           | l2             | True | 1   | 0.0074 | 0.1771    |
| ALL_SKILLS           | ip             | False| 1   | 0.0074 | 0.1771    |
| ALL_SKILLS           | ip             | True | 1   | 0.0074 | 0.1771    |

This experiment studies how well the skills related to a given occupation can be linked from the user's description of their occupation. The results of the evaluation in this specific case seem to imply that in order to get all the skills related to a given occupation it would be much more efficient to link to the occupation and then get the related skills directly from the database. This extremely low recall is determined by the fact that there are now many more true positives than before (an average of 27 skills per occupation) and that our values of k end up being too small to find a significant amount of them. Moreover, the simple description of the job doesn't seem to be sufficient to find many related skills.

As for the hyperparameters, the following is true:

1. The results obtained without MMR are better than the results obtained without MMR. 
2. Different retrieval functions return the same results. 
3. The best embedding methods is LABEL_AND_DESCRIPTION, although the resulting recall is still pretty low. This might happen because the description of the skill could contain information related to jobs to which the skill can be applied to.

In [ ]:
# Best results are given without MMR and using only the preferred label for inputs
def evaluate(model: TextEmbeddingModel, database: pd.DataFrame, test_set: pd.DataFrame, target_column: str, k: int) -> Tuple[float, float]:
    """Returns the precision and recall at k for the vector search through the embedding generated 
    by a given model on a provided dataset. The model must be a TextEmbeddingModel,
    the database must be a Dataframe containing the columns 'PREFERREDLABEL' and 'CODE',
    while the test set must be a Dataframe containing the columns 'synthetic_query' and
    target_column.

    Args:
        model (TextEmbeddingModel): model to be evaluated.
        database (pd.DataFrame): database from which the data should be retrieved.
        test_set (pd.DataFrame): test set for the evaluation.
        target_column (str): name of the target column with the true positives.
            The content should be a list of strings.
        k (int): k for the accuracy at k to be evaluated.

    Returns:
        Tuple[float, float]: precision and recall at k
    """
    assert k<=100
    # Check that the elements of the target column are lists
    assert all(test_set[target_column].apply(lambda x: isinstance(x, list)))
    # Calculate embeddings for the database
    db_embeddings = embed_strings_in_batch(list(database['PREFERREDLABEL']), model)
    # Calculate embeddings for the test set
    test_embeddings = embed_strings_in_batch(list(test_set['synthetic_query']), model)
    # Load a collection from chromadb and saves the data
    client = chromadb.Client()
    collection = client.create_collection(name='eval')
    collection.add(
        documents = list(database['PREFERREDLABEL']),
        metadatas = [{"CODE": row[target_column]} for _, row in database.iterrows()],
        embeddings = db_embeddings,
        ids = [f"id_{i}" for i in range(len(database))]
    )
    # Save the data retrieved from the test set
    vector_search_results = []
    for test_embedding in test_embeddings:
        # Find the top 100 documents and save them in vector_search_results
        documents_from_search = collection.query(query_embeddings=test_embedding, n_results=100, include=["metadatas", "embeddings"])
        vector_search_results.append([elem["CODE"] for elem in documents_from_search["metadatas"][0]])
    # Evaluate precision and recall at k 
    return precision_at_k(vector_search_results, list(test_set[target_column]), k), recall_at_k(vector_search_results, list(test_set[target_column]), k)
